In [191]:
import csv
import sqlite3

In [ ]:
connection = sqlite3.connect("all_hiski_records.sqlite3")
cursor = connection.cursor()

In [193]:
header = []

with open('Moving_record_parishes_with_formats_v2.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    cols = reader.fieldnames[:19]
    

col_to_idx = {cols[i]: i for i in range(len(cols))}
display(col_to_idx)

{'parish_normalized': 0,
 'parish': 1,
 'parish_id': 2,
 'images': 3,
 'doc': 4,
 'url': 5,
 'additional': 6,
 'years': 7,
 'source': 8,
 'archive_id': 9,
 'added': 10,
 'Start_year': 11,
 'End_year': 12,
 'Source_type': 13,
 'added_year': 14,
 'kartta': 15,
 'doctype': 16,
 'Notes': 17,
 'Luokittelu': 18}

In [ ]:
books = []
with open('Moving_record_parishes_with_formats_v2.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    
    for row in reader:
        if row[0] == 'parish_normalized' or row[col_to_idx["Start_year"]] == "" or row[col_to_idx["End_year"]] == "":
            continue

        parish_id = int(row[col_to_idx["parish_id"]])
        parish_normalized = row[col_to_idx["parish_normalized"]]
        year_1, year_2 = int(row[col_to_idx["Start_year"]]), int(row[col_to_idx["End_year"]])
        source = row[col_to_idx["source"]]
        archive_id = row[col_to_idx["archive_id"]]
        doc = row[col_to_idx["doc"]]
        luokittelu = row[col_to_idx["Luokittelu"]]


        books.append(
            {
                "parish_id": parish_id, 
                "parish_normalized": parish_normalized, 
                "year_1": year_1, 
                "year_2": year_2, 
                "source": source,
                "archive_id": archive_id,
                "doc": doc,
                "luokittelu": luokittelu
            }
        )

In [ ]:
cursor.execute(f"SELECT event_id, parish_id, arr_year, dep_year FROM emigrated")
emigration_events = cursor.fetchall()
emigration_events_dict = {event[0]: {"parish_id": event[1], "arr_year": event[2], "dep_year": event[3], "books": []} for event in emigration_events}

cursor.execute(f"SELECT event_id, parish_id, arr_year, dep_year FROM immigrated")
immigration_events = cursor.fetchall()
immigration_events_dict = {event[0]: {"parish_id": event[1], "arr_year": event[2], "dep_year": event[3], "books": []} for event in immigration_events}

In [ ]:
for book in books:
    book["events"] = [
        event_id for event_id, event in emigration_events_dict.items()
        if (
            event["parish_id"] == book["parish_id"]
            and (
                (
                    event["dep_year"] != 0
                    and event["dep_year"] >= book["year_1"]
                    and event["dep_year"] <= book["year_2"]
                )
                or (
                    event["arr_year"] != 0
                    and event["arr_year"] >= book["year_1"]
                    and event["arr_year"] <= book["year_2"]
                )
            )
        )
    ] + [
        event_id for event_id, event in immigration_events_dict.items()
        if (
            event["parish_id"] == book["parish_id"]
            and (
                (
                    event["dep_year"] != 0
                    and event["dep_year"] >= book["year_1"]
                    and event["dep_year"] <= book["year_2"]
                )
                or (
                    event["arr_year"] != 0
                    and event["arr_year"] >= book["year_1"]
                    and event["arr_year"] <= book["year_2"]
                )
            )
        )
    ]

In [ ]:
for book in books:
    for event_id in book["events"]:
        if event_id in emigration_events_dict:
            emigration_events_dict[event_id]["books"].append(
                f"{book["parish_normalized"]}/muuttaneet_{book["year_1"]}-{book["year_2"]}_{book["source"]}"
            )
        if event_id in immigration_events_dict:
            immigration_events_dict[event_id]["books"].append(
                f"{book["parish_normalized"]}/muuttaneet_{book["year_1"]}-{book["year_2"]}_{book["source"]}"
            )

In [ ]:
e_events_has_books = [e for e in emigration_events_dict if len(emigration_events_dict[e]["books"]) > 0]
e_events_no_books = [e for e in emigration_events_dict if len(emigration_events_dict[e]["books"]) == 0]

print("portion of events that can be associated with books:", len(e_events_has_books) / len(emigration_events_dict))
print("portion of events that can't be associated with books:", len(e_events_no_books) / len(emigration_events_dict))

portion of events that can be associated with books: 0.6478255379192586
portion of events that can't be associated with books: 0.35217446208074143


In [199]:
books_has_events = [book for book in books if len(book["events"]) > 0]
books_no_events = [book for book in books if len(book["events"]) == 0]

print("portion of books that can be associated with events:", len(books_has_events) / len(books))
print("portion of books that can't be associated with events:", len(books_no_events) / len(books))

portion of books that can be associated with events: 0.10535778496943546
portion of books that can't be associated with events: 0.8946422150305645


In [ ]:
i_events = [[e_id] + list(e.values()) for e_id, e in immigration_events_dict.items()]
e_events = [[e_id] + list(e.values()) for e_id, e in emigration_events_dict.items()]

In [202]:
with open("immigrate_events_with_books.csv", "w") as file:
    writer = csv.writer(file)
    writer.writerows(i_events)

with open("emigrate_events_with_books.csv", "w") as file:
    writer = csv.writer(file)
    writer.writerows(e_events)